# Qdrant 的基础使用方法

# 0、准备

## 0.1 初始化客户端

Qdrant 没有像 Milvus 那样的「Database」概念，Collection 是数据管理的最高层级单位

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PointIdsList,
    PointStruct,
    VectorParams,
)

collection_name = "docs"

client = QdrantClient(url="http://localhost:6333")

# 1、DDL 操作

## 1.1 Collection 相关操作

### ① 查看 Collection 列表

In [2]:
res = client.get_collections()
print(res)


collections = client.get_collections().collections

for coll in collections:
    print(coll.name)

collections=[CollectionDescription(name='clothing_attrs'), CollectionDescription(name='docs'), CollectionDescription(name='pku')]
clothing_attrs
docs
pku


### ② 判断 Collection 是否存在

In [8]:
print(client.collection_exists(collection_name))

True


### ③ 创建 Collection

需要指定向量维度（size）和距离计算方式（distance）

In [4]:
def create_collection(collection_name:str, size:int):
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(
                size=3072,
                distance=Distance.COSINE,
            ),
        )

### ④ 查看 Collection 的信息

In [5]:
from rich import print as rprint

info = client.get_collection(collection_name)

rprint(info)

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    warnings=None,
    indexed_vectors_count=0,
    points_count=0,
    segments_count=8,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=3072,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=1,
            sharding_method=None,
            replication_factor=1,
            write_consistency_factor=1,
            read_fan_out_factor=None,
            read_fan_out_delay_ms=None,
            on_disk_payload=True,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=False,
            payload_m=None,
            inline_storage=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=10000,
            flush_interval_sec=5,
            max_optimization_threads=None,
            prevent_unoptimized=None
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_closed=1),
        quantization_config=None,
        strict_mode_config=None,
        metadata=None
    ),
    payload_schema={},
    update_queue=UpdateQueueInfo(length=0, deferred_points=None)
)

### ⑤ 重建 Collection（清空）

开发过程中如果想重置测试数据，「先删除再创建」是最简单、最可靠的方式

In [7]:
def recreate_collection(name: str, size: int, distance: Distance = Distance.COSINE):
    if client.collection_exists(name):
        client.delete_collection(name)

    create_collection(name, size)


recreate_collection(collection_name, size=3072)


### ⑥ 删除 Collection

In [6]:
client.delete_collection(collection_name)

print(client.collection_exists(collection_name))

False


# 2、DML 操作

## 2.1 初始化嵌入模型

In [9]:
import os
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings

load_dotenv(override=True)

embed_model = init_embeddings(
    model="openai:text-embedding-3-large",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE"),
)

## 2.2 准备 Collection

根据嵌入向量的维度（text-embedding-3-large 为 3072 维）重新创建 Collection

In [10]:
collection_name = "docs"

recreate_collection(collection_name, size=3072)

## 2.3 准备数据

### ① 准备原始数据

In [11]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用程序的开发框架。",
    "Qdrant 是一款用 Rust 实现的高性能开源向量数据库。",
    "RAG 的核心是先检索相关知识，再让大语言模型基于检索结果生成回答。",
    "使用 Docker Compose 可以很方便地在本地启动 Qdrant。",
]

### ② 生成嵌入向量

In [12]:
vectors = embed_model.embed_documents(texts)

### ③ 查看生成的嵌入向量

In [13]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
3072
[-0.030059814453125, 0.0002963542938232422, -0.0281829833984375, 0.01212310791015625, -0.01120758056640625]


### ④ 封装为可写入的数据格式

在 Qdrant 中，写入的数据需要封装为带有 id、vector、payload（元数据）的 PointStruct

In [14]:
points = [
    PointStruct(
        id=i,
        vector=vectors[i],
        payload={
            "text": texts[i],
            "source": "demo",
        },
    )
    for i in range(len(texts))
]

## 2.4 写入数据

### ① 插入数据（upsert）

Qdrant 没有像 Milvus 那样需要手动 flush 的概念，upsert 返回后即代表写入已完成

In [15]:
upsert_res = client.upsert(
    collection_name=collection_name,
    points=points,
)

print("upsert result : ", upsert_res)

upsert result :  operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>


# 3、DQL 操作

## 3.1 遍历数据（scroll）

scroll 相当于 Milvus 中的 query_iterator，可以对大量数据进行分页遍历

In [16]:
points_batch, next_offset = client.scroll(
    collection_name=collection_name,
    limit=20,
    with_payload=True,
    with_vectors=False,
)

for i, point in enumerate(points_batch):
    print(f"第 {i + 1} 条数据：")
    print(f"id : {point.id}, text = {point.payload['text']}, source = {point.payload['source']}")

print("下一页的 offset : ", next_offset)

第 1 条数据：
id : 0, text = LangChain 是一个用于构建 LLM 应用程序的开发框架。, source = demo
第 2 条数据：
id : 1, text = Qdrant 是一款用 Rust 实现的高性能开源向量数据库。, source = demo
第 3 条数据：
id : 2, text = RAG 的核心是先检索相关知识，再让大语言模型基于检索结果生成回答。, source = demo
第 4 条数据：
id : 3, text = 使用 Docker Compose 可以很方便地在本地启动 Qdrant。, source = demo
下一页的 offset :  None


## 3.2 根据主键查询数据（retrieve）

In [17]:
res = client.retrieve(
    collection_name=collection_name,
    ids=[0, 1, 2],
)

print(len(res))

for i, point in enumerate(res):
    print(f"第 {i + 1} 条数据：")
    print(f"id : {point.id}, text = {point.payload['text']}, source = {point.payload['source']}")

3
第 1 条数据：
id : 0, text = LangChain 是一个用于构建 LLM 应用程序的开发框架。, source = demo
第 2 条数据：
id : 1, text = Qdrant 是一款用 Rust 实现的高性能开源向量数据库。, source = demo
第 3 条数据：
id : 2, text = RAG 的核心是先检索相关知识，再让大语言模型基于检索结果生成回答。, source = demo


## 3.3 相似度检索

### ① 准备查询向量

In [18]:
# 相似度检索
query = "什么是向量数据库？"
query_vector = embed_model.embed_query(query)

### ② 执行检索

新版 SDK 推荐使用 query_points() 进行检索（search() 已不推荐使用）

In [19]:
result = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=3,
)

for point in result.points:
    print(f"id : {point.id}, score = {point.score:.4f}, text = {point.payload['text']}")

id : 1, score = 0.5516, text = Qdrant 是一款用 Rust 实现的高性能开源向量数据库。
id : 2, score = 0.1920, text = RAG 的核心是先检索相关知识，再让大语言模型基于检索结果生成回答。
id : 3, score = 0.1564, text = 使用 Docker Compose 可以很方便地在本地启动 Qdrant。


## 3.4 带过滤条件的检索

Qdrant 的一大特点是可以在向量相似度检索的同时，叠加 payload 过滤条件

In [21]:
result = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="demo"),
            )
        ]
    ),
    limit=3,
)

for point in result.points:
    print(f"id : {point.id}, score = {point.score:.4f}, text = {point.payload['text']}")

id : 1, score = 0.5516, text = Qdrant 是一款用 Rust 实现的高性能开源向量数据库。
id : 2, score = 0.1920, text = RAG 的核心是先检索相关知识，再让大语言模型基于检索结果生成回答。
id : 3, score = 0.1564, text = 使用 Docker Compose 可以很方便地在本地启动 Qdrant。


# 4、更新与删除操作

## 4.1 更新 Payload（局部更新）

向量保持不变，只追加或覆盖元数据

In [22]:
client.set_payload(
    collection_name=collection_name,
    payload={"author": "Tom"},
    points=[0],
)

print(client.retrieve(collection_name=collection_name, ids=[0]))

[Record(id=0, payload={'text': 'LangChain 是一个用于构建 LLM 应用程序的开发框架。', 'source': 'demo', 'author': 'Tom'}, vector=None, shard_key=None, order_value=None)]


## 4.2 根据主键删除数据

In [23]:
client.delete(
    collection_name=collection_name,
    points_selector=PointIdsList(points=[3]),
)

print(client.count(collection_name=collection_name, exact=True))

count=3


## 4.3 根据条件（Payload）删除数据

例如：批量删除 source="demo" 的数据

In [24]:
client.delete(
    collection_name=collection_name,
    points_selector=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="demo"),
            )
        ]
    ),
)

print(client.count(collection_name=collection_name, exact=True))

count=0


# 5、开发中常用的其他功能

## 5.1 创建 Payload 索引（加速过滤检索）

生产环境中如果频繁使用 payload 过滤条件，为对应字段建立索引可以大幅提升检索速度。不建索引也可以进行过滤，但数据量增大后速度会明显下降

In [25]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="source",
    field_schema="keyword",
)

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

## 5.2 按条件统计数据量

In [26]:
count = client.count(
    collection_name=collection_name,
    count_filter=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="batch"),
            )
        ]
    ),
    exact=True,
)

print(count)

count=0


## 5.3 批量写入大量数据（upload_points）

当需要写入数千至数万条数据时，相比逐条调用 upsert，使用 upload_points 会自动进行分批与并行上传，效率更高

In [27]:
import random

VECTOR_SIZE = 3072

large_points = (
    PointStruct(
        id=100 + i,
        vector=[random.random() for _ in range(VECTOR_SIZE)],
        payload={"text": f"批量数据 {i}", "source": "batch"},
    )
    for i in range(200)
)

client.upload_points(
    collection_name=collection_name,
    points=large_points,
    batch_size=64,
)

print(client.count(collection_name=collection_name, exact=True))

count=200


# 6、收尾清理

演示结束后，删除 Collection

In [28]:
client.delete_collection(collection_name)

print(client.collection_exists(collection_name))

False
